# MSCAD regeneration pipeline

Regenerates flow records **with flow identifiers** (Src/Dst IP, ports, protocol,
timestamp) from the raw MSCAD PCAPs, verifies the extraction, compares it against the
original Kaggle `MSCAD.csv`, and writes the consolidated CSVs plus a report.

The Kaggle file has the CICFlowMeter features and a `Label` but no identifiers, so
PrefixSpan sequence mining and the attack-state graph cannot be built from it. This
supplies them.

### How this notebook works

Each stage runs as a **subprocess**, not inside the kernel. That is deliberate: the
flow data never enters this kernel's memory, so no capture can crash it no matter how
large, and every stage starts from clean state. Output streams into the cell below as
it runs.

### What you need

Anaconda's `numpy`, `scipy` and `pandas` — all three ship with Anaconda, so normally
there is nothing to install. `scapy` is **not** required for `.pcap` input.
`tshark` is optional: the verifier has its own independent packet counter.

Run the cells in order. Cell 2 is the only one you may need to edit.

## 1. Environment check

Confirms the kernel can run the pipeline before anything touches your data.

In [ ]:
import os, sys, platform, subprocess, importlib

print("Python   :", sys.version.split()[0], "(%s)" % platform.python_version())
print("executable:", sys.executable)
print("platform :", platform.platform())

if sys.version_info < (3, 8):
    print("\n*** Python 3.8+ required ***")

missing = []
for pkg in ("numpy", "scipy", "pandas"):
    try:
        m = importlib.import_module(pkg)
        print("%-8s : %s" % (pkg, getattr(m, "__version__", "?")))
    except ImportError:
        missing.append(pkg)
        print("%-8s : MISSING" % pkg)

for pkg in ("scapy",):
    try:
        m = importlib.import_module(pkg)
        print("%-8s : %s (optional)" % (pkg, getattr(m, "__version__", "?")))
    except ImportError:
        print("%-8s : not installed (optional: only for .pcapng input and "
              "for the synthetic self-test)" % pkg)

if missing:
    print("\nInstall the missing packages into THIS kernel with:")
    print("    %s -m pip install %s" % (sys.executable, " ".join(missing)))
else:
    print("\nAll required packages present.")

## 2. Paths — edit these if your folders differ

`PIPELINE_DIR` must hold `mscad_config.py` and the numbered stage scripts. If you put
this notebook next to them, the default is already right.

In [ ]:
# ---- edit if needed -------------------------------------------------
PIPELINE_DIR = os.getcwd()          # the folder this notebook sits in

PCAP_DIR = r"D:\IDSPROJECT2026\MSCAD\MSCAD Dataset\PCAPs\pcapngtopcap"
CSV_DIR  = r"D:\IDSPROJECT2026\MSCAD\CSV_ORIGINAL"
OUT_DIR  = r"D:\IDSPROJECT2026\MSCAD\MSCAD_REGENERATED"
TSHARK   = r"C:\Program Files\Wireshark\tshark.exe"
# ---------------------------------------------------------------------

need = ["mscad_config.py", "mscad_flowmeter.py", "01_pcap_to_csv.py"]
absent = [f for f in need if not os.path.isfile(os.path.join(PIPELINE_DIR, f))]
if absent:
    print("*** Not found in PIPELINE_DIR = %s" % PIPELINE_DIR)
    for f in absent:
        print("      missing:", f)
    print("\nSet PIPELINE_DIR above to the folder holding the pipeline scripts")
    print("(the MSCAD_PIPELINE folder of the repo), then re-run this cell.")
else:
    print("pipeline dir : %s   OK" % PIPELINE_DIR)

for label, p in (("PCAP dir", PCAP_DIR), ("CSV dir", CSV_DIR)):
    print("%-13s: %s   %s" % (label, p, "EXISTS" if os.path.isdir(p) else "*** MISSING ***"))
print("%-13s: %s   %s" % ("out dir", OUT_DIR,
                          "exists" if os.path.isdir(OUT_DIR) else "will be created"))
print("%-13s: %s   %s" % ("tshark", TSHARK,
                          "found" if os.path.isfile(TSHARK) else "not found (optional)"))

OPTS = ["--pcap-dir", PCAP_DIR, "--csv-dir", CSV_DIR, "--out-dir", OUT_DIR,
        "--tshark", TSHARK]


def run_stage(script, extra=()):
    """Run one stage as a subprocess, streaming its output into this cell.

    The stage runs in its own process, so flow data never enters this kernel
    and a large capture cannot kill it.
    """
    cmd = [sys.executable, "-u", os.path.join(PIPELINE_DIR, script)]
    cmd += list(extra) + OPTS
    print("$ %s %s %s\n" % (os.path.basename(sys.executable), script,
                            " ".join(extra)))
    p = subprocess.Popen(cmd, cwd=PIPELINE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("\n[exit code %d]" % p.returncode)
    if p.returncode != 0:
        print("*** This stage reported a problem. Read the error above; "
              "nothing was skipped silently.")
    return p.returncode


print("\nrun_stage() ready.")

## 3. Optional: self-test on synthetic captures

Proves the whole pipeline works in this kernel without touching your data. Needs
`scapy`. Writes only into a temp folder. Skip it if you have already run
`selftest.bat`.

In [ ]:
RUN_SELFTEST = False          # set True to run

if RUN_SELFTEST:
    import tempfile
    T = os.path.join(tempfile.gettempdir(), "mscad_selftest")
    subprocess.run([sys.executable, "-u",
                    os.path.join(PIPELINE_DIR, "make_test_fixture.py"), T],
                   cwd=PIPELINE_DIR)
    t_opts = ["--pcap-dir", os.path.join(T, "pcaps", "pcapngtopcap"),
              "--csv-dir", os.path.join(T, "csv_original"),
              "--out-dir", os.path.join(T, "out")]
    for st in ("00_inspect_inputs.py", "01_pcap_to_csv.py", "02_verify.py",
               "03_compare_with_kaggle.py", "04_consolidate.py",
               "05_report.py"):
        print("\n" + "=" * 70 + "\n" + st + "\n" + "=" * 70)
        subprocess.run([sys.executable, "-u",
                        os.path.join(PIPELINE_DIR, st)] + t_opts,
                       cwd=PIPELINE_DIR)
    print("\nSelf-test done. Look for 'NO PROBLEMS FOUND' in the "
          "02_verify output above.")
else:
    print("skipped (set RUN_SELFTEST = True to run it)")

## 4. Stage 00 — inspect the inputs

Checks both input folders and lists them. If the PCAP folder is missing it searches
`D:\IDSPROJECT2026\MSCAD` for a folder named `pcapngtopcap` and reports which one it
used. Then prints the original Kaggle file's shape, every column name, and its Label
counts.

Reads only — it never modifies, moves or deletes anything in the input folders.

In [ ]:
run_stage("00_inspect_inputs.py")

## 5. Stage 01a — N-0 only (the quick test)

**Check this before the full run.** Confirm that:

- `packets read` minus `IPv4 packets in flows` equals the ARP/IPv6 count, and nothing else
- the feature column count matches your Kaggle file (66)
- no column is reported as unmappable
- `Flow Duration` looks like microseconds (tens of thousands for a short session, not hundredths)

In [ ]:
run_stage("01_pcap_to_csv.py", ["N-0"])

Peek at the N-0 output before committing to the full run:

In [ ]:
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

n0 = os.path.join(OUT_DIR, "raw", "N-0_flows.csv")
if os.path.isfile(n0):
    df = pd.read_csv(n0)
    print("shape:", df.shape)
    cols = ["Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol", "Timestamp",
            "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts",
            "Fwd Pkt Len Min", "Fwd Pkt Len Max", "Flow IAT Mean",
            "Fwd Header Len", "Init Bwd Win Byts", "Label"]
    cols = [c for c in cols if c in df.columns]
    print(df[cols].head(10).to_string())
    print("\npacket total in flows:",
          int(df["Tot Fwd Pkts"].sum() + df["Tot Bwd Pkts"].sum()))
    print("protocols:", dict(df["Protocol"].value_counts()))
else:
    print("not found yet:", n0)

## 6. Stage 01b — all six captures

Expect roughly a minute or two in total, not the 10–20 you may be budgeting: a
704,000-packet brute-force capture measured 19.5 s at a flat 53 MB of memory.

Progress prints every 100,000 packets. Each attack capture is processed twice — once
to find the attacker IPs, once to write the labelled flows — so that nothing has to be
held in memory. The top 5 senders are printed per capture so you can check the
attacker detection yourself.

In [ ]:
run_stage("01_pcap_to_csv.py")

## 7. Stage 02 — verify the extraction (Part 2)

The one check that matters most is the first table: `IPv4 in flows` must equal the
independent counts **exactly**. If it does, no packet was lost or double-counted.

Everything in the sanity table should be 0 except `inf` (zero-duration flows' rate
features — normal CICFlowMeter behaviour), `1-packet flows` and
`identical feature rows` (both expected, both counted rather than dropped).

In [ ]:
run_stage("02_verify.py")

## 8. Stage 03 — compare with the original Kaggle MSCAD.csv (Part 3)

Schema, rows per label, the microseconds units check, per-feature Kolmogorov–Smirnov
statistics per label, and row-level matching on the integer-feature key.

Two things to read carefully:

- **Section 3.4.** If the four `Subflow` columns top the KS table, set
  `SUBFLOW_MODE = "copy"` in `mscad_config.py` and re-run stage 01. If the
  `Std`/`Var` columns or `Down/Up Ratio` lead it, check `STD_DDOF` and
  `JAVA_INTEGER_DIVISION`.
- **Section 3.5.** A low match rate is expected and is *not* evidence the extraction
  is wrong — stage 02's packet reconciliation is the completeness check. Read
  section 4.6 of the generated report for why.

In [ ]:
run_stage("03_compare_with_kaggle.py")

## 9. Stage 04 — the consolidated CSVs (Part 4)

- `MSCAD_consolidated.csv` — identifiers + features + `Label` + `Label_Kaggle` +
  `Source_Capture`, sorted by `Timestamp`. **This is the file for PrefixSpan and the
  attack-state graph.**
- `MSCAD_consolidated_ML.csv` — features + `Label` only, infinities dropped and
  identical feature rows de-duplicated, for RF / XGBoost / LSTM / CNN.

In [ ]:
run_stage("04_consolidate.py")

## 10. Stage 05 — the report (Part 5)

Writes `MSCAD_regeneration_report.md` into the output folder and copies every script
there, so the run is reproducible. The report contains the method, all the tables
above, the file descriptions and a paragraph you can adapt for your thesis
methodology.

In [ ]:
run_stage("05_report.py")

## 11. Load the result and sanity-check it here

In [ ]:
cons = os.path.join(OUT_DIR, "MSCAD_consolidated.csv")
mlf = os.path.join(OUT_DIR, "MSCAD_consolidated_ML.csv")

if os.path.isfile(cons):
    full = pd.read_csv(cons, low_memory=False)
    print("MSCAD_consolidated.csv :", full.shape)
    print("\nlabels:")
    print(full["Label"].value_counts().to_string())
    print("\nper capture:")
    print(full["Source_Capture"].value_counts().to_string())
    print("\nLabel_Kaggle filled on %d rows"
          % int(full["Label_Kaggle"].notna().sum()))
    print("\ntime span: %s  ->  %s"
          % (full["Timestamp"].min(), full["Timestamp"].max()))
    print("\nfirst rows:")
    print(full[["Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol",
                "Timestamp", "Label", "Source_Capture"]].head(8).to_string())
else:
    print("not found yet:", cons)

if os.path.isfile(mlf):
    ml = pd.read_csv(mlf, low_memory=False)
    print("\nMSCAD_consolidated_ML.csv :", ml.shape)
    print(ml["Label"].value_counts().to_string())
    print("\nNOTE: dropping infinite rows removes every zero-duration flow,")
    print("which in a port scan is most of the scan, and de-duplicating")
    print("identical feature vectors removes real flood/scan/brute-force")
    print("traffic. Compare the two label tables above before you train on")
    print("the ML file -- the class balance is not the same.")

## 12. Next step: the sequence view

This is what the regeneration was for. `MSCAD_consolidated.csv` can be grouped into
per-host-pair sequences of labelled flows, which is the input shape PrefixSpan wants.

In [ ]:
if os.path.isfile(cons):
    seq = (full.sort_values("Timestamp")
               .groupby(["Src IP", "Dst IP"])["Label"]
               .apply(list))
    print("host pairs with at least one flow :", len(seq))
    multi = seq[seq.map(len) > 1]
    print("host pairs with a multi-flow sequence:", len(multi))
    print("\nlongest 5 sequences (truncated to 12 flows for display):")
    for (s, d), labs in multi.sort_values(key=lambda x: x.map(len),
                                          ascending=False).head(5).items():
        shown = labs[:12]
        print("  %-16s -> %-16s  n=%-6d  %s%s"
              % (s, d, len(labs), " > ".join(shown),
                 " > ..." if len(labs) > 12 else ""))
else:
    print("run stage 04 first")